# Node Diffusion Training (Kaggle)
文本条件 + 图结构条件的节点坐标扩散生成

In [ ]:
import os
REPO_DIR = '/kaggle/working/PlanDiffusion_wzm'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b wzm-shenzhou https://github.com/WeeZHnMin/PlanDiffusion_wzm.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull origin wzm-shenzhou')
print('repo ready:', REPO_DIR)

In [ ]:
# 路径配置
DATA_PATH = '/kaggle/input/datasets/weizhiming/plandiffusion-layout/layout_dataset.npz'
VOCAB_DIR = '/kaggle/working/PlanDiffusion_wzm/node_diffusion/unified_vocab'
SAVE_DIR  = '/kaggle/working/checkpoints/node_diffusion'
RESUME    = ''  # 续训时填写路径，例如 '/kaggle/working/checkpoints/node_diffusion/latest.pt'

# 训练超参数
BATCH_SIZE     = 256
TOTAL_STEPS    = 300000
LR             = 1e-4
WEIGHT_DECAY   = 1e-4
LOG_INTERVAL   = 100
SAVE_INTERVAL  = 5000
TIMESTEPS      = 1000

# 模型超参数
MODEL_CHANNELS = 384
NUM_LAYERS     = 6
NUM_HEADS      = 6

In [ ]:
import os, math, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
os.makedirs(SAVE_DIR, exist_ok=True)

# 从词表配置读取 BPE 词表大小
vocab_cfg      = json.loads(open(os.path.join(VOCAB_DIR, 'vocab_config.json'), encoding='utf-8').read())
BPE_VOCAB_SIZE = vocab_cfg['bpe_vocab_size']
print(f'BPE vocab size: {BPE_VOCAB_SIZE}')

# HuggingFace token（从 Kaggle Secrets 读取）
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    HF_REPO  = 'weizhiming/plandiffusion-node-diffusion'
    print(f'HF token loaded, will upload to {HF_REPO}')
except Exception:
    HF_TOKEN = None
    print('HF token not found, checkpoint upload disabled')

## 模型定义

In [ ]:
def timestep_embedding(timesteps, dim):
    half = dim // 2
    freqs = torch.exp(
        -math.log(10000) * torch.arange(half, dtype=torch.float32, device=timesteps.device) / half
    )
    args = timesteps[:, None].float() * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)


def attention(q, k, v, d_k, mask=None, dropout=None):
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask.unsqueeze(1) == 1, -1e4)
    scores = F.softmax(scores.float(), dim=-1).to(q.dtype)
    if dropout is not None:
        scores = dropout(scores)
    return torch.matmul(scores, v)


class MultiHeadAttention(nn.Module):
    def __init__(self, heads, d_model, dropout=0.1):
        super().__init__()
        self.d_k = d_model // heads
        self.h = heads
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        bs = q.size(0)
        q = self.q_linear(q).view(bs, -1, self.h, self.d_k).transpose(1, 2)
        k = self.k_linear(k).view(bs, -1, self.h, self.d_k).transpose(1, 2)
        v = self.v_linear(v).view(bs, -1, self.h, self.d_k).transpose(1, 2)
        out = attention(q, k, v, self.d_k, mask, self.dropout)
        out = out.transpose(1, 2).contiguous().view(bs, -1, self.h * self.d_k)
        return self.out(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_model * 2)
        self.linear2 = nn.Linear(d_model * 2, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))


class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.adj_attn    = MultiHeadAttention(heads, d_model, dropout)
        self.global_attn = MultiHeadAttention(heads, d_model, dropout)
        self.ff      = FeedForward(d_model, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, adj_mask, pad_mask, text_kv=None, text_key_mask=None):
        x2 = self.norm1(x)
        if text_kv is not None:
            kv = torch.cat([x2, text_kv], dim=1)
            node_key_pad = pad_mask[:, :1, :]
            text_key_pad = text_key_mask.unsqueeze(1)
            combined_mask = torch.cat(
                [node_key_pad.expand(-1, x.shape[1], -1),
                 text_key_pad.expand(-1, x.shape[1], -1)], dim=2)
            global_out = self.global_attn(x2, kv, kv, combined_mask)
        else:
            global_out = self.global_attn(x2, x2, x2, pad_mask)
        x = (x
             + self.dropout(self.adj_attn(x2, x2, x2, adj_mask))
             + self.dropout(global_out))
        x2 = self.norm2(x)
        x = x + self.dropout(self.ff(x2))
        return x


class NodeDiffusionTransformer(nn.Module):
    def __init__(self, model_channels=256, num_layers=6, num_heads=4,
                 dropout=0.1, bpe_vocab_size=12000):
        super().__init__()
        self.model_channels = model_channels
        self.time_embed = nn.Sequential(
            nn.Linear(model_channels, model_channels),
            nn.SiLU(),
            nn.Linear(model_channels, model_channels),
        )
        self.input_emb  = nn.Linear(2, model_channels)
        self.text_embed = nn.Embedding(bpe_vocab_size, model_channels, padding_idx=0)
        self.layers = nn.ModuleList(
            [EncoderLayer(model_channels, num_heads, dropout) for _ in range(num_layers)]
        )
        self.output_head = nn.Sequential(
            nn.Linear(model_channels, model_channels),
            nn.ReLU(),
            nn.Linear(model_channels, model_channels // 2),
            nn.Linear(model_channels // 2, 2),
        )
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'NodeDiffusionTransformer: {n:,} parameters')

    def _build_masks(self, adj_matrix, node_mask):
        adj_mask = 1 - adj_matrix
        pad_keys = (1 - node_mask).unsqueeze(1)
        adj_mask = torch.clamp(adj_mask + pad_keys, 0, 1)
        pad_mask = pad_keys.expand_as(adj_mask)
        return adj_mask, pad_mask

    def forward(self, x, timesteps, adj_matrix, node_mask,
                prompt_tokens=None, prompt_mask=None, **kwargs):
        x = x.permute(0, 2, 1).float()
        t_emb = self.time_embed(timestep_embedding(timesteps, self.model_channels)).unsqueeze(1)
        out = self.input_emb(x) + t_emb
        adj_mask, pad_mask = self._build_masks(adj_matrix.float(), node_mask.float())
        text_kv = text_key_mask = None
        if prompt_tokens is not None:
            text_kv = self.text_embed(prompt_tokens)
            text_key_mask = (1 - prompt_mask.float()) if prompt_mask is not None else (prompt_tokens == 0).float()
        for layer in self.layers:
            out = layer(out, adj_mask, pad_mask, text_kv, text_key_mask)
        out = self.output_head(out)
        return out.permute(0, 2, 1)

## 扩散过程

In [ ]:
class GaussianDiffusion:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02):
        self.T = timesteps
        betas           = torch.linspace(beta_start, beta_end, timesteps)
        alphas          = 1.0 - betas
        alphas_bar      = torch.cumprod(alphas, dim=0)
        alphas_bar_prev = torch.cat([torch.tensor([1.0]), alphas_bar[:-1]])
        self.betas                     = betas
        self.alphas                    = alphas
        self.alphas_bar                = alphas_bar
        self.alphas_bar_prev           = alphas_bar_prev
        self.sqrt_alphas_bar           = alphas_bar.sqrt()
        self.sqrt_one_minus_alphas_bar = (1 - alphas_bar).sqrt()
        self.posterior_variance = (betas * (1 - alphas_bar_prev) / (1 - alphas_bar)).clamp(min=1e-20)

    def _to(self, device):
        for attr in ['betas','alphas','alphas_bar','alphas_bar_prev',
                     'sqrt_alphas_bar','sqrt_one_minus_alphas_bar','posterior_variance']:
            setattr(self, attr, getattr(self, attr).to(device))
        return self

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        s1 = self.sqrt_alphas_bar[t].view(-1, 1, 1)
        s2 = self.sqrt_one_minus_alphas_bar[t].view(-1, 1, 1)
        return s1 * x0 + s2 * noise, noise

    def training_losses(self, model, x0, t, model_kwargs):
        self._to(x0.device)
        x0    = x0.float()
        noise = torch.randn_like(x0)
        xt, _ = self.q_sample(x0, t, noise)
        pred_noise = model(xt, t, **model_kwargs)
        mask = model_kwargs['node_mask'].float().unsqueeze(1)
        loss = ((pred_noise - noise) ** 2 * mask).sum() / (mask.sum() * 2 + 1e-8)
        with torch.no_grad():
            s1 = self.sqrt_alphas_bar[t].view(-1, 1, 1)
            s2 = self.sqrt_one_minus_alphas_bar[t].view(-1, 1, 1)
            pred_x0    = (xt - s2 * pred_noise) / s1
            raw_mse    = ((pred_x0 - x0) ** 2 * mask).sum() / (mask.sum() * 2 + 1e-8)
            coord_rmse = raw_mse.sqrt().item() * 160.0
        return loss, coord_rmse

## 数据集

In [ ]:
class NodeDataset(Dataset):
    def __init__(self, npz_path):
        d = np.load(npz_path, allow_pickle=True)
        self.coords        = d['node_coords'].astype(np.float32)
        self.adj_matrix    = d['adj_matrix'].astype(np.float32)
        self.node_mask     = d['node_mask'].astype(np.float32)
        self.prompt_tokens = d['prompt_tokens'].astype(np.int64)
        self.prompt_lens   = d['prompt_lens'].astype(np.int32)
        T = self.prompt_tokens.shape[1]
        self.prompt_mask = np.zeros((len(self.prompt_tokens), T), dtype=np.float32)
        for i, l in enumerate(self.prompt_lens):
            self.prompt_mask[i, :l] = 1.0
        print(f'NodeDataset: {len(self.coords)} samples')

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        x = self.coords[idx].T.copy()
        cond = {
            'adj_matrix':    self.adj_matrix[idx],
            'node_mask':     self.node_mask[idx],
            'prompt_tokens': self.prompt_tokens[idx],
            'prompt_mask':   self.prompt_mask[idx],
        }
        return torch.from_numpy(x), {k: torch.from_numpy(v) for k, v in cond.items()}


def make_loader(npz_path, batch_size):
    ds = NodeDataset(npz_path)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True, pin_memory=True)
    def infinite():
        while True:
            yield from loader
    return infinite()

## 初始化模型

In [ ]:
model = NodeDiffusionTransformer(
    model_channels=MODEL_CHANNELS,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    bpe_vocab_size=BPE_VOCAB_SIZE,
).to(device)

# 双卡并行
if torch.cuda.device_count() > 1:
    print(f'使用 {torch.cuda.device_count()} 张 GPU')
    model = torch.nn.DataParallel(model)

diffusion = GaussianDiffusion(timesteps=TIMESTEPS)
opt       = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler    = torch.cuda.amp.GradScaler()  # FP16 梯度缩放

start_step = 0
if RESUME:
    ckpt = torch.load(RESUME, map_location=device)
    model.load_state_dict(ckpt['model'])
    opt.load_state_dict(ckpt['opt'])
    scaler.load_state_dict(ckpt['scaler'])
    start_step = ckpt['step'] + 1
    print(f'resumed from step {start_step}')

data = make_loader(DATA_PATH, BATCH_SIZE)

## 训练循环

In [ ]:
def upload_to_hf(local_path, filename):
    if HF_TOKEN is None:
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.create_repo(HF_REPO, repo_type='model', exist_ok=True, private=True)
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=filename,
            repo_id=HF_REPO,
            repo_type='model',
        )
        print(f'  uploaded -> hf:{HF_REPO}/{filename}')
    except Exception as e:
        print(f'  HF upload failed: {e}')


model.train()
running_loss = 0.0
running_rmse = 0.0

for step in range(start_step, TOTAL_STEPS):
    x, cond = next(data)
    x    = x.to(device)
    cond = {k: v.to(device) for k, v in cond.items()}

    t = torch.randint(0, TIMESTEPS, (x.shape[0],), device=device)

    opt.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        loss, coord_rmse = diffusion.training_losses(model, x, t, cond)

    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt)
    scaler.update()

    running_loss += loss.item()
    running_rmse += coord_rmse

    if step % LOG_INTERVAL == 0:
        avg      = running_loss / LOG_INTERVAL if step > 0 else running_loss
        avg_rmse = running_rmse / LOG_INTERVAL if step > 0 else running_rmse
        running_loss = running_rmse = 0.0
        print(f'step {step:6d} | loss {avg:.4f} | coord_rmse {avg_rmse:.2f} px')

    if step > 0 and step % SAVE_INTERVAL == 0:
        path = os.path.join(SAVE_DIR, 'latest.pt')
        torch.save({
            'model':  model.state_dict(),
            'opt':    opt.state_dict(),
            'scaler': scaler.state_dict(),
            'step':   step,
        }, path)
        print(f'  saved -> {path}')
        upload_to_hf(path, 'latest.pt')

torch.save({
    'model':  model.state_dict(),
    'opt':    opt.state_dict(),
    'scaler': scaler.state_dict(),
    'step':   TOTAL_STEPS,
}, os.path.join(SAVE_DIR, 'model_final.pt'))
upload_to_hf(os.path.join(SAVE_DIR, 'model_final.pt'), 'model_final.pt')
print('训练完成')